# 3 · Supervised Fine-Tuning

In [1]:
import json
from pathlib import Path

from datasets import load_dataset
from tokenizers import Tokenizer
from transformers import PreTrainedTokenizerFast

DATASET = "Pondsiders/tinystories-gpt4-instruct"
CONTEXT = 512
ARTIFACTS = Path("artifacts")

/Users/jefferyharrell/Pondside/Workshop/Projects/lil-transformy-2/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
raw = json.loads((ARTIFACTS / "tokenizer.json").read_text())

for entry in raw["added_tokens"]:
    if entry["content"] == "<|reserved_3|>":
        entry["content"] = "<|pad|>"
vocab = raw["model"]["vocab"]
if "<|reserved_3|>" in vocab:
    vocab["<|pad|>"] = vocab.pop("<|reserved_3|>")

tokenizer = PreTrainedTokenizerFast(
    tokenizer_object=Tokenizer.from_str(json.dumps(raw)),
    bos_token="<|endoftext|>",
    eos_token="<|im_end|>",
    pad_token="<|pad|>",
)

for name in ["<|endoftext|>", "<|im_start|>", "<|im_end|>", "<|pad|>"]:
    print(f"{tokenizer.convert_tokens_to_ids(name):4d}  {name}")

   0  <|endoftext|>
   1  <|im_start|>
   2  <|im_end|>
   3  <|pad|>


In [3]:
CHAT_TEMPLATE = (
    "{{ '<|endoftext|>' }}"
    "{% for message in messages %}"
    "{{ '<|im_start|>' + message['role'] + '\n' + message['content'] + '<|im_end|>' + '\n' }}"
    "{% endfor %}"
    "{% if add_generation_prompt %}{{ '<|im_start|>assistant\n' }}{% endif %}"
)
tokenizer.chat_template = CHAT_TEMPLATE

messages = [{"role": "user", "content": "Tell me a story about a duck named Pondside."}]
print(tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False))

<|endoftext|><|im_start|>user
Tell me a story about a duck named Pondside.<|im_end|>
<|im_start|>assistant



In [4]:
dataset = load_dataset(DATASET)
print(dataset)
print()
row = dataset["train"][0]
print(row["prompt"])
print(row["story"][:80] + "...")

DatasetDict({
    train: Dataset({
        features: ['prompt', 'story', 'source_id', 'template_id', 'names', 'kind', 'all_names'],
        num_rows: 50000
    })
    validation: Dataset({
        features: ['prompt', 'story', 'source_id', 'template_id', 'names', 'kind', 'all_names'],
        num_rows: 1000
    })
})

I'd like a story about a boy named Tim, please.
One day, a little boy named Tim was eager to play outside. He saw that the sun w...


In [5]:
BOS = tokenizer.convert_tokens_to_ids("<|endoftext|>")
IM_START = tokenizer.convert_tokens_to_ids("<|im_start|>")
IM_END = tokenizer.convert_tokens_to_ids("<|im_end|>")
PAD = tokenizer.convert_tokens_to_ids("<|pad|>")


def encode(text):
    return tokenizer.encode(text, add_special_tokens=False)


def assemble(prompt, story):
    request = (
        [BOS, IM_START] + encode("user\n" + prompt) + [IM_END]
        + encode("\n") + [IM_START] + encode("assistant\n")
    )
    response = encode(story) + [IM_END]
    return request, response


request, response = assemble(row["prompt"], row["story"])
print(tokenizer.decode(request), "…")
print(f"request {len(request)} tokens, response {len(response)} tokens")

<|endoftext|><|im_start|>user
I'd like a story about a boy named Tim, please.<|im_end|>
<|im_start|>assistant
 …
request 25 tokens, response 169 tokens


In [6]:
# Training must see exactly what inference will see: the request half of every
# training example has to match the chat template's rendering token for token.
mismatches = 0
for pair in dataset["validation"]:
    request, _ = assemble(pair["prompt"], pair["story"])
    templated = tokenizer.apply_chat_template(
        [{"role": "user", "content": pair["prompt"]}],
        add_generation_prompt=True,
    )["input_ids"]
    if request != templated:
        mismatches += 1
print(f"hand assembly vs chat template: {mismatches} mismatches in {len(dataset['validation'])} pairs")

hand assembly vs chat template: 0 mismatches in 1000 pairs


In [7]:
lengths = []
for pair in dataset["train"]:
    request, response = assemble(pair["prompt"], pair["story"])
    lengths.append(len(request) + len(response))

import numpy as np
lengths = np.array(lengths)
print(f"median {int(np.median(lengths))} tokens · p95 {int(np.percentile(lengths, 95))} · max {lengths.max()}")
print(f"over {CONTEXT}: {(lengths > CONTEXT).sum():,} of {len(lengths):,} ({100 * (lengths > CONTEXT).mean():.2f}%)")

median 204 tokens · p95 285 · max 1020
over 512: 60 of 50,000 (0.12%)


In [8]:
import numpy as np

IGNORE = -100


def build_example(pair):
    request, response = assemble(pair["prompt"], pair["story"])
    length = len(request) + len(response)
    if length > CONTEXT:
        return None
    pad = CONTEXT - length
    input_ids = request + response + [PAD] * pad
    labels = [IGNORE] * len(request) + response + [IGNORE] * pad
    attention_mask = [1] * length + [0] * pad
    return input_ids, labels, attention_mask


def build_split(split):
    examples = [build_example(pair) for pair in split]
    kept = [e for e in examples if e is not None]
    ids, labels, mask = (np.array(t, dtype=np.int16) for t in zip(*kept))
    print(f"{len(kept):,} examples kept, {len(examples) - len(kept):,} dropped for length")
    return ids, labels, mask


train_ids, train_labels, train_mask = build_split(dataset["train"])
valid_ids, valid_labels, valid_mask = build_split(dataset["validation"])

49,940 examples kept, 60 dropped for length
997 examples kept, 3 dropped for length


In [9]:
# The pad token and -100 wear different hats but must agree: anywhere the
# input is padding, the label must be IGNORE, or the model learns to predict
# nothing and the loss looks wonderful while the model gets worse.
assert (train_labels[train_ids == PAD] == IGNORE).all()
assert (valid_labels[valid_ids == PAD] == IGNORE).all()

# And the graded region must be exactly the response: story plus its <|im_end|>.
graded = train_labels[0][train_labels[0] != IGNORE].astype(np.int64)
print(tokenizer.decode(graded)[:120] + " …")
print(f"…ends with: {tokenizer.convert_ids_to_tokens([int(graded[-1])])}")

One day, a little boy named Tim was eager to play outside. He saw that the sun was shining and the snow was melting. He  …
…ends with: ['<|im_end|>']


In [10]:
example = train_ids[0].astype(np.int64)
markers = ["·" if label == IGNORE else "█" for label in train_labels[0]]
tokens = tokenizer.convert_ids_to_tokens(example)
print("".join(markers[:80]))
print()
for token, marker in list(zip(tokens, markers))[:24]:
    print(f"  {marker}  {token}")

·························███████████████████████████████████████████████████████

  ·  <|endoftext|>
  ·  <|im_start|>
  ·  us
  ·  er
  ·  Ċ
  ·  I
  ·  'd
  ·  Ġlike
  ·  Ġa
  ·  Ġstory
  ·  Ġabout
  ·  Ġa
  ·  Ġboy
  ·  Ġnamed
  ·  ĠTim
  ·  ,
  ·  Ġplease
  ·  .
  ·  <|im_end|>
  ·  Ċ
  ·  <|im_start|>
  ·  ass
  ·  ist
  ·  ant


In [11]:
total = train_mask.sum()
graded = (train_labels != IGNORE).sum()
padding = (train_ids == PAD).sum()
print(f"tokens in play: {total:,}")
print(f"graded: {graded:,} ({100 * graded / total:.0f}% of real tokens)")
print(f"padding: {padding:,} ({100 * padding / train_ids.size:.0f}% of all positions)")

tokens in play: 10,493,205
graded: 9,367,757 (89% of real tokens)
padding: 15,076,075 (59% of all positions)


In [12]:
import math
import torch
from transformers import LlamaForCausalLM

BATCH_SIZE = 64
PEAK_LR = 6e-5
WARMUP_STEPS = 50
TRUNK_EPOCHS = 3
ANNEAL_STEPS = 100
GRAD_CLIP = 1.0
SEED = 20260831

STEPS_PER_EPOCH = len(train_ids) // BATCH_SIZE
TRUNK_STEPS = TRUNK_EPOCHS * STEPS_PER_EPOCH

# Checkpoints peel off the trunk where these many pairs have been seen.
MILESTONES = [1_000, 5_000, 20_000, len(train_ids)]

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
torch.manual_seed(SEED)
rng = np.random.default_rng(SEED)

model = LlamaForCausalLM.from_pretrained(ARTIFACTS / "lil-transformy-2").to(device)
model.train()

optimizer = torch.optim.AdamW(
    model.parameters(), lr=PEAK_LR, betas=(0.9, 0.95), weight_decay=0.1,
    fused=(device == "cuda"),
)

print(f"{device} · {STEPS_PER_EPOCH} steps/epoch · trunk {TRUNK_STEPS} steps · "
      f"milestones at pairs {MILESTONES}")

Loading weights:   0%|          | 0/74 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 74/74 [00:00<00:00, 7303.54it/s]

mps · 780 steps/epoch · trunk 2340 steps · milestones at pairs [1000, 5000, 20000, 49940]


In [13]:
# Warmup-Stable-Decay: warm up, then hold the trunk FLAT. There is no decay
# here — every checkpoint peeled off the trunk gets its own short anneal
# instead, so one run can father many finished models.
def trunk_lr(step):
    if step < WARMUP_STEPS:
        return PEAK_LR * (step + 1) / WARMUP_STEPS
    return PEAK_LR


def batches(epoch_seed):
    order = np.random.default_rng(epoch_seed).permutation(len(train_ids))
    for start in range(0, len(order) - BATCH_SIZE + 1, BATCH_SIZE):
        rows = order[start : start + BATCH_SIZE]
        yield {
            "input_ids": torch.from_numpy(train_ids[rows].astype(np.int64)).to(device),
            "attention_mask": torch.from_numpy(train_mask[rows].astype(np.int64)).to(device),
            "labels": torch.from_numpy(train_labels[rows].astype(np.int64)).to(device),
        }


@torch.no_grad()
def validation_loss(n_batches=8):
    model.eval()
    total = 0.0
    for start in range(0, n_batches * BATCH_SIZE, BATCH_SIZE):
        batch = {
            "input_ids": torch.from_numpy(valid_ids[start : start + BATCH_SIZE].astype(np.int64)).to(device),
            "attention_mask": torch.from_numpy(valid_mask[start : start + BATCH_SIZE].astype(np.int64)).to(device),
            "labels": torch.from_numpy(valid_labels[start : start + BATCH_SIZE].astype(np.int64)).to(device),
        }
        total += model(**batch).loss.item()
    model.train()
    return total / n_batches

In [14]:
# Smoke test: the machinery, not the training. A few dozen steps on whatever
# hardware is present — the loss must fall, a checkpoint must peel and save.
SMOKE_STEPS = 30

print(f"validation loss before a single step: {validation_loss():.4f}")

step = 0
for batch in batches(epoch_seed=SEED):
    if step >= SMOKE_STEPS:
        break
    for group in optimizer.param_groups:
        group["lr"] = trunk_lr(step)
    loss = model(**batch).loss
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
    optimizer.step()
    if step % 10 == 0:
        print(f"step {step:3d} · lr {trunk_lr(step):.2e} · loss {loss.item():.4f}")
    step += 1

print(f"validation loss after {SMOKE_STEPS} steps: {validation_loss():.4f}")

validation loss before a single step: 1.4017


step   0 · lr 1.20e-06 · loss 1.4432


step  10 · lr 1.32e-05 · loss 1.3483


step  20 · lr 2.52e-05 · loss 1.3281


validation loss after 30 steps: 1.1473


In [15]:
# Peel-and-anneal rehearsal: save a trunk checkpoint, anneal a copy a few
# steps, save the annealed model with the tokenizer riding along.
SMOKE_DIR = ARTIFACTS / "smoke-checkpoint"

model.save_pretrained(SMOKE_DIR / "trunk")

annealed = LlamaForCausalLM.from_pretrained(SMOKE_DIR / "trunk").to(device)
annealed.train()
anneal_opt = torch.optim.AdamW(annealed.parameters(), lr=PEAK_LR, betas=(0.9, 0.95), weight_decay=0.1)

n = 10
for i, batch in enumerate(batches(epoch_seed=SEED + 1)):
    if i >= n:
        break
    for group in anneal_opt.param_groups:
        group["lr"] = PEAK_LR * (n - i) / n
    loss = annealed(**batch).loss
    anneal_opt.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(annealed.parameters(), GRAD_CLIP)
    anneal_opt.step()

annealed.save_pretrained(SMOKE_DIR / "annealed")
tokenizer.save_pretrained(SMOKE_DIR / "annealed")

for f in sorted((SMOKE_DIR / "annealed").iterdir()):
    print(f.name)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 27.24it/s]

Loading weights:   0%|          | 0/74 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 74/74 [00:00<00:00, 8485.39it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 38.32it/s]

chat_template.jinja
config.json
generation_config.json
model.safetensors
tokenizer.json
tokenizer_config.json
